In [1]:
from pathlib import Path
import sys

import pandas as pd


# ============================================================
# PROJECT CONFIG
# ============================================================

current = Path.cwd().resolve()

PROJECT_ROOT = next(
    p for p in [current, *current.parents]
    if (p / "src" / "data_pipeline" / "config.py").exists()
)

sys.path.insert(
    0,
    str(PROJECT_ROOT / "src" / "data_pipeline")
)

from config import (
    INSIDERS_CSV,
    GROUND_TRUTH_PARQUET,
    GROUND_TRUTH_CSV,
)


# ============================================================
# INPUT CHECK
# ============================================================

if not INSIDERS_CSV.exists():
    raise FileNotFoundError(
        f"Ground-truth source not found:\n{INSIDERS_CSV}"
    )

print("✓ Ground-truth source found.")


# ============================================================
# LOAD ORIGINAL CERT GROUND TRUTH
# ============================================================

df = pd.read_csv(
    INSIDERS_CSV,
    dtype=str,
)

df["dataset"] = pd.to_numeric(
    df["dataset"],
    errors="coerce",
)

df["scenario"] = pd.to_numeric(
    df["scenario"],
    errors="coerce",
).astype("Int64")


# Keep CERT r4.2 only
gt = df[
    df["dataset"] == 4.2
].copy()


# ============================================================
# PARSE INCIDENT TIMESTAMPS
# ============================================================

gt["attack_start"] = pd.to_datetime(
    gt["start"],
    format="%m/%d/%Y %H:%M:%S",
    errors="coerce",
)

gt["attack_end"] = pd.to_datetime(
    gt["end"],
    format="%m/%d/%Y %H:%M:%S",
    errors="coerce",
)


# ============================================================
# ADD CLEAN METADATA
# ============================================================

scenario_names = {
    1: "After-hours + USB + Web exfiltration",
    2: "Long-term USB data theft before departure",
    3: "Keylogger + impersonation + disruptive email",
}

gt["scenario_name"] = (
    gt["scenario"]
    .map(scenario_names)
)


# Scenarios 1 and 2 are relevant to
# the data-exfiltration experiments.
gt["exfiltration_relevant"] = (
    gt["scenario"]
    .isin([1, 2])
    .astype(int)
)

gt["source"] = "CERT_r4.2"

# Original CERT incidents are not synthetic.
gt["is_synthetic"] = 0


# ============================================================
# INCIDENT DURATION
# ============================================================

gt["duration_days"] = (
    gt["attack_end"]
    - gt["attack_start"]
).dt.total_seconds() / 86400


# ============================================================
# FINAL CLEAN SCHEMA
# ============================================================

gt = gt[
    [
        "user",
        "scenario",
        "scenario_name",
        "details",
        "attack_start",
        "attack_end",
        "duration_days",
        "exfiltration_relevant",
        "source",
        "is_synthetic",
    ]
].sort_values(
    [
        "scenario",
        "attack_start",
    ]
).reset_index(
    drop=True
)


# ============================================================
# VALIDATION
# ============================================================

duplicate_rows = gt.duplicated(
    subset=[
        "user",
        "scenario",
        "attack_start",
    ]
).sum()

missing_values = int(
    gt.isna().sum().sum()
)

scenario_counts = (
    gt["scenario"]
    .value_counts()
    .sort_index()
)


assert len(gt) == 70, (
    f"Unexpected incident count: {len(gt)}"
)

assert gt["user"].nunique() == 70, (
    "Unexpected number of malicious users."
)

assert scenario_counts.to_dict() == {
    1: 30,
    2: 30,
    3: 10,
}, (
    f"Unexpected scenario distribution: "
    f"{scenario_counts.to_dict()}"
)

assert int(
    gt["exfiltration_relevant"].sum()
) == 60, (
    "Unexpected exfiltration-relevant count."
)

assert missing_values == 0, (
    f"Ground truth contains "
    f"{missing_values} missing values."
)

assert duplicate_rows == 0, (
    f"Ground truth contains "
    f"{duplicate_rows} duplicate rows."
)

assert (
    gt["attack_start"]
    <= gt["attack_end"]
).all(), (
    "Found attack_start later than attack_end."
)

assert set(
    gt["scenario"].dropna().astype(int)
) == {1, 2, 3}

assert (
    gt["source"] == "CERT_r4.2"
).all()

assert (
    gt["is_synthetic"] == 0
).all()


# ============================================================
# SAVE CLEAN GROUND TRUTH
# ============================================================

gt.to_parquet(
    GROUND_TRUTH_PARQUET,
    index=False,
)

gt.to_csv(
    GROUND_TRUTH_CSV,
    index=False,
)


# ============================================================
# FINAL STATUS
# ============================================================

print("\n" + "=" * 70)
print("✅ BUILD GROUND TRUTH: PASS")
print("=" * 70)

print(
    f"\nTotal incidents        : "
    f"{len(gt)}"
)

print(
    f"Unique malicious users : "
    f"{gt['user'].nunique()}"
)

print(
    f"Exfiltration relevant  : "
    f"{gt['exfiltration_relevant'].sum()}"
)

print(
    "\nScenario counts:"
)

for scenario, count in scenario_counts.items():
    print(
        f"  Scenario {scenario}: "
        f"{count}"
    )

print(
    f"\nParquet saved to:\n"
    f"{GROUND_TRUTH_PARQUET}"
)

print(
    f"\nCSV saved to:\n"
    f"{GROUND_TRUTH_CSV}"
)

✓ Ground-truth source found.

✅ BUILD GROUND TRUTH: PASS

Total incidents        : 70
Unique malicious users : 70
Exfiltration relevant  : 60

Scenario counts:
  Scenario 1: 30
  Scenario 2: 30
  Scenario 3: 10

Parquet saved to:
C:\Users\HP\Documents\FYP\Data_engineering\data\processed\ground_truth_r42.parquet

CSV saved to:
C:\Users\HP\Documents\FYP\Data_engineering\data\processed\ground_truth_r42.csv


In [3]:
# ============================================================
# GROUND TRUTH INSPECTION
# ============================================================

print("=" * 70)
print("GROUND TRUTH SUMMARY")
print("=" * 70)


print(
    f"\nTotal incidents        : "
    f"{len(gt)}"
)

print(
    f"Unique malicious users : "
    f"{gt['user'].nunique()}"
)


# ============================================================
# SCENARIO DISTRIBUTION
# ============================================================

scenario_summary = (
    gt["scenario"]
    .value_counts()
    .sort_index()
    .rename_axis("scenario")
    .reset_index(name="incident_count")
)

print("\nScenario distribution:")
display(scenario_summary)


# ============================================================
# EXFILTRATION RELEVANCE
# ============================================================

exfiltration_summary = (
    gt["exfiltration_relevant"]
    .value_counts()
    .sort_index()
    .rename_axis("exfiltration_relevant")
    .reset_index(name="incident_count")
)

print("\nExfiltration relevance:")
display(exfiltration_summary)


# ============================================================
# INCIDENT DURATION
# ============================================================

duration_summary = (
    gt
    .groupby("scenario")[
        "duration_days"
    ]
    .agg(
        [
            "count",
            "min",
            "median",
            "mean",
            "max",
        ]
    )
    .round(2)
    .reset_index()
)

print("\nDuration statistics by scenario:")
display(duration_summary)


# ============================================================
# DATA INTEGRITY
# ============================================================

missing_summary = (
    gt
    .isna()
    .sum()
    .rename("missing_values")
    .reset_index()
    .rename(
        columns={
            "index": "column"
        }
    )
)

duplicates = gt.duplicated(
    subset=[
        "user",
        "scenario",
        "attack_start",
    ]
).sum()


print("\nMissing values:")
display(missing_summary)

print(
    f"\nDuplicate incidents : "
    f"{duplicates}"
)


# ============================================================
# SAMPLE
# ============================================================

print("\nFirst five incidents:")

display(
    gt.head()
)


print("\n" + "=" * 70)
print("✅ GROUND TRUTH INSPECTION COMPLETE")
print("=" * 70)

GROUND TRUTH SUMMARY

Total incidents        : 70
Unique malicious users : 70

Scenario distribution:


,scenario,incident_count
0,1,30
1,2,30
2,3,10



Exfiltration relevance:


,exfiltration_relevant,incident_count
0,0,10
1,1,60



Duration statistics by scenario:


,scenario,count,min,median,mean,max
0,1,30,0.01,6.19,5.29,10.86
1,2,30,44.08,56.19,55.13,58.46
2,3,10,1.26,1.41,1.40,1.48



Missing values:


,column,missing_values
0,user,0
1,scenario,0
2,scenario_name,0
3,details,0
4,attack_start,0
5,attack_end,0
6,duration_days,0
7,exfiltration_relevant,0
8,source,0
9,is_synthetic,0



Duplicate incidents : 0

First five incidents:


,user,scenario,scenario_name,details,attack_start,attack_end,duration_days,exfiltration_relevant,source,is_synthetic
0,KPC0073,1,After-hours + USB + Web exfiltration,r4.2-1-KPC0073.csv,2010-07-07 20:05:29,2010-07-15 03:08:16,7.293600,1,CERT_r4.2,0
1,RKD0604,1,After-hours + USB + Web exfiltration,r4.2-1-RKD0604.csv,2010-07-13 20:04:53,2010-07-20 03:36:55,6.313912,1,CERT_r4.2,0
2,BIH0745,1,After-hours + USB + Web exfiltration,r4.2-1-BIH0745.csv,2010-07-13 20:15:23,2010-07-13 21:20:44,0.045382,1,CERT_r4.2,0
3,JMB0308,1,After-hours + USB + Web exfiltration,r4.2-1-JMB0308.csv,2010-07-14 00:56:15,2010-07-21 01:46:49,7.035116,1,CERT_r4.2,0
4,EHB0824,1,After-hours + USB + Web exfiltration,r4.2-1-EHB0824.csv,2010-07-22 21:48:43,2010-07-29 01:08:41,6.138866,1,CERT_r4.2,0



✅ GROUND TRUTH INSPECTION COMPLETE
